# Chapter 06 · Linux commands for MLOps: hands-on

Companion to `notes/06_linux_commands.html` (video 01:31:55 – 01:54:09).

Every command the instructor runs on the EC2 Ubuntu server, run here for real inside a throw-away **`sandbox/`** folder.

* Cells starting with `%%bash` run in a bash shell. Each cell is a new shell, but they all start in the sandbox folder.
* This notebook works on **macOS and Linux**. The few commands that only exist on Ubuntu servers (`apt`, `useradd`) are explained, and the notebook checks whether they're available.
* To follow along on the real server instead, launch EC2 (Chapter 05) and type the same commands there.

In [1]:
import os, shutil, subprocess

# All work happens inside ./sandbox, which is wiped each time this cell runs
SANDBOX = os.path.abspath("sandbox")
shutil.rmtree(SANDBOX, ignore_errors=True)
os.makedirs(SANDBOX)
os.chdir(SANDBOX)            # %%bash cells inherit this working directory
for k in ("CLICOLOR", "CLICOLOR_FORCE", "LS_COLORS"):
    os.environ.pop(k, None)  # plain (uncoloured) ls output
print("working in:", os.getcwd())

working in: /Users/hemanthreddy/Desktop/data_science/MLOps/projects/ch06_linux_commands/sandbox


## 1 · `sudo` and package updates  (video 01:32)

On a fresh Ubuntu server the first thing to run is:

```bash
sudo apt update          # refresh the package list (sudo = run as admin/root)
sudo apt-get update      # same job, older tool; preferred in scripts/Dockerfiles
sudo apt upgrade -y      # actually install newer versions (beyond the video)
```

Let's check what OS this notebook is running on and whether `apt` exists here:

In [2]:
%%bash
uname -s -m
command -v apt >/dev/null && echo "apt is available: this is Debian/Ubuntu" \
  || echo "apt not found: not Ubuntu (on macOS the equivalent is Homebrew). Run 'sudo apt update' on EC2."

Darwin arm64


apt not found: not Ubuntu (on macOS the equivalent is Homebrew). Run 'sudo apt update' on EC2.


## 2 · Looking around: `ls`, `ls -al`, `pwd`  (video 01:34)

A fresh home folder looks empty with `ls`, but `ls -al` also shows **hidden** files (names starting with `.`).
Let's create one visible file and one hidden file to see the difference.

In [3]:
%%bash
touch visible.txt .hidden_config
echo "--- ls ---";      ls
echo "--- ls -al ---";  ls -al
echo "--- pwd (print working directory) ---"; pwd

--- ls ---


visible.txt


--- ls -al ---


total 0
drwxr-xr-x  4 hemanthreddy  staff  128 Sep 16 20:18 .
drwxr-xr-x  4 hemanthreddy  staff  128

 Sep 16 20:18 ..
-rw-r--r--  1 hemanthreddy  staff    0 Sep 16 20:18 .hidden_config
-rw-r--r--  1 he

manthreddy  staff    0 Sep 16 20:18 visible.txt


--- pwd (print working directory) ---
/Users/hemanthreddy/Desktop/data_science/MLOps/projects/ch06_l

inux_commands/sandbox


## 3 · Folders: `mkdir`, `cd`, `cd ..`, `rmdir`  (video 01:35)

In [4]:
%%bash
mkdir test
ls
cd test && echo "inside: $(pwd)" && ls -l   # empty folder
cd ..   && echo "back in: $(pwd)"
rmdir test                                  # rmdir only works on EMPTY folders
ls
mkdir bappy_test                            # the folder used for the rest of the demo
ls -d */

test
visible.txt


inside: /Users/hemanthreddy/Desktop/data_science/MLOps/projects/ch06_linux_commands/sandbox/test


total 0


back in: /Users/hemanthreddy/Desktop/data_science/MLOps/projects/ch06_linux_commands/sandbox


visible.txt


bappy_test/


In [5]:
%%bash
# rmdir refuses a non-empty folder; rm -r deletes a folder and its contents
mkdir -p not_empty && touch not_empty/file.txt
rmdir not_empty || echo "↑ rmdir failed as expected"
rm -r not_empty && echo "rm -r removed it"

rmdir: not_empty: Directory not empty


↑ rmdir failed as expected


rm -r removed it


## 4 · Files: `touch`, editing, `cat`  (video 01:38)

The instructor edits files with **vim**: `vim bappy.txt` → press `i` (insert mode) → type → `Esc` → `:wq` → Enter.
Vim is interactive and can't run inside a notebook, so here we write the file with `printf` and use `cat` to read it back.

In [6]:
%%bash
touch bappy.txt
printf 'hello\n' > bappy.txt     # what you'd type in vim
cat bappy.txt

hello


## 5 · Running a Python file  (video 01:39)

In [7]:
%%bash
cat > test.py <<'EOF'
print("hello")
EOF
cat test.py
python3 test.py

print("hello")


hello


## 6 · Copy, move, delete: `cp`, `mv`, `rm`  (video 01:40)

`cp` leaves the original in place. `mv` removes it from the source (and also renames files). `rm` deletes permanently; there's no recycle bin.

In [8]:
%%bash
cp test.py bappy_test/
echo "after cp → here: $(ls | tr '\n' ' ') | bappy_test: $(ls bappy_test)"
rm bappy_test/test.py
mv test.py bappy_test/
echo "after mv → here: $(ls | tr '\n' ' ') | bappy_test: $(ls bappy_test)"
rm bappy_test/test.py
echo "after rm → bappy_test: '$(ls bappy_test)'"
mv bappy.txt notes.txt && mv notes.txt bappy.txt && echo "mv also renames"

after cp → here: bappy_test bappy.txt test.py visible.txt  | bappy_test: test.py


after mv → here: bappy_test bappy.txt visible.txt  | bappy_test: test.py


after rm → bappy_test: ''


mv also renames


## 7 · Help and processes: `man`, `top`  (video 01:42)

`man ls` opens the manual (press `q` to quit). `top` is Linux's Task Manager and is interactive (`Ctrl+C` or `q` to quit), so here we print one snapshot of it.

In [9]:
%%bash
man ls 2>/dev/null | col -b | sed -n '1,12p' || ls --help | head -12

LS(1)			    General Commands Manual			 LS(1)

NAME
     ls – list directory contents

SYNOPSIS
   

  ls [-@ABCFGHILOPRSTUWabcdefghiklmnopqrstuvwxy1%,] [--color=when]
	[-D format] [file ...]

DESCRIPT

ION
     For each operand that names a file of a type other than directory, ls
     displays its nam

e as well as any requested, associated information.  For


In [10]:
%%bash
if [ "$(uname)" = "Darwin" ]; then
  top -l 1 -n 5 -stats pid,command,cpu,mem | head -20   # macOS flags
else
  top -b -n 1 | head -15                                # Linux flags (what you'd use on EC2)
fi
echo "--- non-interactive alternative: ps ---"
ps aux | head -5

Processes: 528 total, 3 running, 1 stuck, 524 sleeping, 3246 threads 
2026/09/16 20:18:49
Load Avg: 

6.62, 6.74, 5.55 
CPU usage: 20.0% user, 20.40% sys, 59.60% idle 
SharedLibs: 191M resident, 47M dat

a, 43M linkedit.
MemRegions: 738423 total, 1533M resident, 46M private, 522M shared.
PhysMem: 7461M 

used (1549M wired, 3371M compressor), 162M unused.
VM: 271T vsize, 6144M framework vsize, 111158370(

0) swapins, 117106407(0) swapouts.
Networks: packets: 75925235/81G in, 33958231/28G out.
Disks: 2766

...

05  39.1  0.3 435492640  25312   ??  U     6Sep26 2439:38.02 /System/Library/PrivateFrameworks/iClou

dDriveCore.framework/Versions/A/Support/bird
hemanthreddy       855  33.7  0.3 435427248  24512   ??

  S     2Sep26 1331:35.60 /System/Library/PrivateFrameworks/FileProvider.framework/Support/fileprovi

derd
_windowserver      476  16.1  0.4 436537232  36992   ??  Ss    2Sep26 682:22.40 /System/Library

/PrivateFrameworks/SkyLight.framework/Resources/WindowServer -daemon


## 8 · Permissions: `ls -l` and `chmod`  (video 01:43)

The permission string has 3 groups (owner / group / others), each made of `r`=4, `w`=2, `x`=1.

In [11]:
%%bash
ls -l bappy.txt
chmod 700 bappy.txt && ls -l bappy.txt     # owner rwx, nobody else anything
chmod 664 bappy.txt && ls -l bappy.txt     # the Ubuntu default in the video: rw-rw-r--
chmod 644 bappy.txt && ls -l bappy.txt     # typical for normal files

-rw-r--r--  1 hemanthreddy  staff  6 Sep 16 20:18 bappy.txt


-rwx------  1 hemanthreddy  staff  6 Sep 16 20:18 bappy.txt


-rw-rw-r--  1 hemanthreddy  staff  6 Sep 16 20:18 bappy.txt


-rw-r--r--  1 hemanthreddy  staff  6 Sep 16 20:18 bappy.txt


In [12]:
def to_octal(perm: str) -> str:
    """'rw-rw-r--' -> '664'"""
    perm = perm[-9:]
    val = {"r": 4, "w": 2, "x": 1, "-": 0}
    return "".join(str(sum(val[c] for c in perm[i:i + 3])) for i in (0, 3, 6))

def to_symbolic(octal: str) -> str:
    """'755' -> 'rwxr-xr-x'"""
    return "".join(("r" if d & 4 else "-") + ("w" if d & 2 else "-") + ("x" if d & 1 else "-")
                   for d in map(int, octal))

for mode, use in [("700", "video example: owner only"), ("664", "video default"), ("755", "scripts & folders"),
                  ("644", "normal files"), ("600", "private files"), ("400", "SSH .pem key")]:
    print(f"chmod {mode}  ->  {to_symbolic(mode)}   ({use})")
assert to_octal("-rw-rw-r--") == "664"

chmod 700  ->  rwx------   (video example: owner only)
chmod 664  ->  rw-rw-r--   (video default)
chmod 755  ->  rwxr-xr-x   (scripts & folders)
chmod 644  ->  rw-r--r--   (normal files)
chmod 600  ->  rw-------   (private files)
chmod 400  ->  r--------   (SSH .pem key)


In [13]:
%%bash
# execute permission in action: a script can't run until it has +x
printf '#!/bin/bash\necho "script ran"\n' > run.sh
./run.sh 2>&1 || echo "↑ no execute permission yet"
chmod +x run.sh && ./run.sh

bash: line 3: ./run.sh: Permission denied


↑ no execute permission yet


script ran


## 9 · Archives: `tar` and `zip`  (video 01:47)

The instructor's first attempt failed because `-f` has to be followed by the **archive name**. The correct forms:

In [14]:
%%bash
printf 'print("hello")\n' > test.py
tar -cvf archive.tar test.py bappy.txt        # c=create v=verbose f=file name
echo "--- list contents (t) ---"
tar -tvf archive.tar
mkdir -p extracted && tar -xvf archive.tar -C extracted   # x=extract (-C = into this folder)
ls extracted
echo "--- gzip-compressed (beyond the video) ---"
tar -czvf archive.tar.gz test.py bappy.txt
ls -l archive.tar archive.tar.gz

a test.py


a bappy.txt


--- list contents (t) ---


-rw-r--r--  0 hemanthreddy staff      15 Sep 16 20:18 test.py
-rw-r--r--  0 hemanthreddy staff      

 6 Sep 16 20:18 bappy.txt

...

--- gzip-compressed (beyond the video) ---


a test.py


a bappy.txt


-rw-r--r--  1 hemanthreddy  staff  3072 Sep 16 20:18 archive.tar
-rw-r--r--  1 hemanthreddy  staff  

 172 Sep 16 20:18 archive.tar.gz


In [15]:
%%bash
if command -v zip >/dev/null; then
  zip files.zip test.py bappy.txt && unzip -l files.zip
else
  echo "zip not installed (on Ubuntu: sudo apt install -y zip unzip)"
fi

  adding: test.py

 (stored 0%)
  adding: bappy.txt (stored 0%)


Archive:  files.zip


  Length      Date    Time    Name
---------  ---------- -----   ----


       15  09-16-2026 20:18   test.py
        6  09-16-2026 20:18   bappy.txt
---------             

        -------
       21                     2 files


## 10 · Users and utilities: `whoami`, `date`, `echo`, `head`, `tail`  (video 01:50)

User management (`sudo useradd bappy`, `sudo userdel bappy`) needs root on a Linux server, so it isn't run here.
The instructor also says it isn't needed for this course.

In [16]:
%%bash
whoami
date
echo "hello from echo"

hemanthreddy


Wed Sep 16 20:18:50 EDT 2026


hello from echo


In [17]:
%%bash
# head / tail are most useful on long files such as logs
for i in $(seq 1 30); do echo "2026-09-16 10:$(printf %02d $i) INFO request $i served"; done > app.log
echo "request 17 failed" | sed 's/^/2026-09-16 10:17 ERROR /' >> app.log
echo "--- head -n 3 ---"; head -n 3 app.log
echo "--- tail -n 3 ---"; tail -n 3 app.log
echo "--- grep ERROR (beyond the video) ---"; grep ERROR app.log

--- head -n 3 ---


2026-09-16 10:01 INFO request 1 served
2026-09-16 10:02 INFO request 2 served
2026-09-16 10:03 INFO 

request 3 served


--- tail -n 3 ---


2026-09-16 10:29 INFO request 29 served
2026-09-16 10:30 INFO request 30 served
2026-09-16 10:17 ERR

OR request 17 failed


--- grep ERROR (beyond the video) ---


2026-09-16 10:17 ERROR request 17 failed


## 11 · Extra commands for deployment (beyond the video)

These come up constantly when you serve a model from a server.

In [18]:
%%bash
echo "--- disk space ---"; df -h . | head -3
echo "--- folder sizes ---"; du -sh * | sort -h | tail -5
echo "--- environment variables ---"
export MODEL_NAME=emotion-xgb && echo "MODEL_NAME=$MODEL_NAME"
echo "--- is a process running? ---"
ps aux | grep -i "[j]upyter" | head -2 | cut -c1-120
echo "--- HTTP check (what you'd do against your API) ---"
curl -s -o /dev/null -w "github.com answered HTTP %{http_code}\n" https://github.com || echo "no network"

--- disk space ---


Filesystem      Size    Used   Avail Capacity iused ifree %iused  Mounted on
/dev/disk3s1   460Gi   

119Gi   313Gi    28%    1.3M  3.3G    0%   /System/Volumes/Data


--- folder sizes ---


4.0K	bappy.txt
4.0K	files.zip
4.0K	run.sh
4.0K	test.py
8.0K	extracted


...

hemanthreddy     17414   5.2  1.0 435347552  82528   ??  S     8:18PM   0:01.04 /Library/Frameworks/

Python.framework/Ver
hemanthreddy     17415   0.0  0.0 435304560   1008   ??  S     8:18PM   0:00.00

 /bin/zsh -c source /Users/hemanthreddy/.


--- HTTP check (what you'd do against your API) ---


github.com answered HTTP 200


## 12 · Summary

| Task | Command |
|---|---|
| update packages (Ubuntu) | `sudo apt update` |
| where am I / what's here | `pwd`, `ls`, `ls -al` |
| folders | `mkdir`, `cd`, `cd ..`, `rmdir`, `rm -r` |
| files | `touch`, `vim` (i → Esc → `:wq`), `cat`, `head`, `tail` |
| run code | `python3 file.py` |
| copy / move / delete | `cp`, `mv`, `rm` |
| help / processes | `man`, `top`, `ps aux` |
| permissions | `ls -l`, `chmod 700`, `chmod +x` |
| archives | `tar -cvf` / `-xvf` / `-czvf`, `zip` |
| misc | `whoami`, `date`, `echo` |

The `sandbox/` folder can be deleted at any time; re-running the first cell recreates it.